In [1]:
import pylab as pl
import numpy as np
import scipy as sp
import scipy.optimize as spo
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import csv
import tensorflow as tf
from tensorflow import keras

In [2]:
#train the LSTM
#begin by setting up two batches for training data
batch_x_train = []
batch_y_train = []

#the first half is linear
#a 'zero' corresponds to linear
df_1 = pd.read_csv('np_lin_100.csv')

for index, rows in df_1.iterrows():
    batch_y_train.append(0)
    list_1 = rows
    batch_x_train.append(list_1)

#the other half is circular
#a 'one' corresponds to circular
df_2 = pd.read_csv('np_circ_100.csv')            

for index, rows in df_2.iterrows():
    batch_y_train.append(1)
    list_2 = rows
    batch_x_train.append(list_2)

#first, convert the x training data into a tensor
#second, convert the y training into a categorical tensor
batch_x_train = tf.convert_to_tensor(batch_x_train, dtype=tf.float32) 
batch_y_train = tf.keras.utils.to_categorical(batch_y_train, num_classes=2)
batch_y_val = batch_y_train

#next, create a set to validate against.
batch_x_val = []

df_3 = pd.read_csv('np_t_lin_100.csv', usecols=range(30))

for index, rows in df_3.iterrows():
    list_3 = rows
    batch_x_val.append(list_3)
    
df_4 = pd.read_csv('np_t_circ_100.csv', usecols=range(30))
for index, rows in df_4.iterrows():
    list_4 = rows
    batch_x_val.append(list_4)

#as before, convert it to a tensor
batch_x_val = tf.convert_to_tensor(batch_x_val, dtype=tf.float32)

#then set the training parameters
lr = 1e-2
num_nodes = 6
num_classes = 2
num_epochs = 400
timesteps = 71
tf.keras.backend.clear_session()
X = tf.keras.Input(name='X', shape=[batch_x_train.shape[1],1], dtype=tf.dtypes.float32)
lstm_output = tf.keras.layers.LSTM(num_nodes)(X)
prediction = tf.keras.layers.Dense(num_classes)(lstm_output)
opt = tf.keras.optimizers.Adam(learning_rate=lr)
model = keras.Model(inputs=X, outputs=prediction)
model.compile(optimizer=opt, loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),metrics=['accuracy'])
model.fit(x=batch_x_train, y=batch_y_train, epochs=num_epochs, validation_data=(batch_x_val,batch_y_val))

/opt/anaconda3/lib/python3.13/site-packages/tensorflow/python/framework/constant_op.py:108: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return ops.EagerTensor(value, ctx.device_name, dtype)


NameError: name 'batch_x_test' is not defined

In [ ]:
#since we have training and validation sets,
#we can proceed with a testing set

batch_x_test = []
df_3 = pd.read_csv('np_lin_300.csv', usecols=range(30))
for index, rows in df_3.iterrows():
    list_3 = rows
    batch_x_test.append(list_3)
    
df_4 = pd.read_csv('np_circ_300.csv', usecols=range(30))
for index, rows in df_4.iterrows():
    list_4 = rows
    batch_x_test.append(list_4)
    
batch_x_test = tf.convert_to_tensor(batch_x_test, dtype=tf.float32)

In [ ]:
#next, we generate a confusion matrix
y_prediction = model.predict(batch_x_test)
y_prediction = np.argmax(y_prediction, axis=1)
y_test = np.argmax(batch_y_train, axis=1)
result = tf.math.confusion_matrix(y_test, y_prediction, num_classes=2, weights=None)
labels = ["linear", "circular"]
cm = confusion_matrix(y_test, y_prediction)

In [ ]:
#create a display to evaluate the results
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Blues)
plt.show()